In [4]:
%pip install torch

  Using cached setuptools-84.2.1-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.3/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.5/124.1 MB 1.9 MB/s eta 0:01:07
   ---------------------------------------- 1.0/124.1 MB 2.1 MB/s eta 0:00:59
    --------------------------------------- 1.6/124.1 MB 2.1 MB/s eta 0:00:58
    --------------------------------------- 1.8/124.1 MB 1.9 MB/s eta 0:01:06
    --------------------------------------- 2.1/124.1 MB 1.9 MB/s eta 0:01:05
    --------------------------------------- 2.6/124.1 MB 1.9 MB/s eta 0:01:04
   - -------------------------------------- 3.7/124.1 MB 2.3 MB/s eta


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
%pip install scikit-learn

  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached narwhals-2.27.1-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.7.0-py3-none-any.whl.metadata (24 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.3 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.3 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.3 MB 799.2 kB/s eta 0:00:10
   ------ --------------------------------- 1.3/8.3 MB 1.6 MB/s eta 0:00:05
   -------- ------------------------------- 1.8/8.3 MB 1.8 MB/s eta 0:00:04
   ----------- ---------------------------- 2.4/8.3 MB 2.0 MB/s eta 0:00:03
   ------------- -------------------------- 2.9/8.3 MB 2.1 MB/s eta 0:00:03
   ----------------- ---------------------- 3.7/8.3 MB 2.3 MB/s eta 0:00:03
   --------------------- ------------------ 4.5/8.3 MB 2.5 MB/s eta 0:


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:

import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

from collections import Counter


In [10]:
import os

print("Current working directory:")
print(os.getcwd())

print("\nFiles in the current directory:")
print(os.listdir())

Current working directory:
c:\Users\mengj\OneDrive\Desktop\GENAI_LLM_Engineering_SCFP125016\Activity_2

Files in the current directory:
['IMDB Dataset (1).csv', 'TextClassificationWithLSTM.ipynb']


In [12]:


df = pd.read_csv("IMDB Dataset (1).csv")

print("Dataset loaded successfully!")

print("\nDataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 reviews:")
display(df.head())

print("\nMissing values:")
print(df.isnull().sum())

print("\nSentiment distribution:")
print(df["sentiment"].value_counts())



Dataset loaded successfully!

Dataset shape: (50000, 2)

Column names:
['review', 'sentiment']

First 5 reviews:


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive



Missing values:
review       0
sentiment    0
dtype: int64

Sentiment distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [14]:

# Check missing values
print("Missing values before cleaning:")
print(df.isnull().sum())

# Remove rows with missing values
df = df.dropna(subset=["review", "sentiment"])

# Remove duplicate reviews
df = df.drop_duplicates(subset=["review"])

# Remove extra spaces from reviews
df["review"] = df["review"].astype(str).str.strip()

# Normalize sentiment labels
df["sentiment"] = df["sentiment"].str.strip().str.lower()

# Keep only positive and negative reviews
df = df[df["sentiment"].isin(["positive", "negative"])]

print("\nDataset shape after cleaning:", df.shape)

print("\nMissing values after cleaning:")
print(df.isnull().sum())

print("\nSentiment distribution after cleaning:")
print(df["sentiment"].value_counts())

print("\nSample reviews after cleaning:")
display(df.head())



Missing values before cleaning:
review       0
sentiment    0
dtype: int64

Dataset shape after cleaning: (49582, 2)

Missing values after cleaning:
review       0
sentiment    0
dtype: int64

Sentiment distribution after cleaning:
sentiment
positive    24884
negative    24698
Name: count, dtype: int64

Sample reviews after cleaning:


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [15]:

# Convert sentiment labels into numerical values
df["sentiment"] = df["sentiment"].map({
    "negative": 0,
    "positive": 1
})

print("Sentiment labels converted successfully!")

print("\nFirst 5 rows:")
display(df.head())

print("\nSentiment distribution:")
print(df["sentiment"].value_counts())

print("\nLabel meanings:")
print("0 = Negative")
print("1 = Positive")



Sentiment labels converted successfully!

First 5 rows:


,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1



Sentiment distribution:
sentiment
1    24884
0    24698
Name: count, dtype: int64

Label meanings:
0 = Negative
1 = Positive


In [16]:

# Separate reviews and sentiment labels
X = df["review"]
y = df["sentiment"]

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Dataset split successfully!")

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining sentiment distribution:")
print(y_train.value_counts())

print("\nTesting sentiment distribution:")
print(y_test.value_counts())



Dataset split successfully!

Training samples: 39665
Testing samples: 9917

Training sentiment distribution:
sentiment
1    19907
0    19758
Name: count, dtype: int64

Testing sentiment distribution:
sentiment
1    4977
0    4940
Name: count, dtype: int64


In [17]:

# Text preprocessing
def clean_text(text):
    text = text.lower()
    text = re.sub(r"<br\s*/?>", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


# Clean training and testing reviews
X_train_clean = X_train.apply(clean_text)
X_test_clean = X_test.apply(clean_text)

print("Text cleaning completed!")

print("\nOriginal review:")
print(X_train.iloc[0][:300])

print("\nCleaned review:")
print(X_train_clean.iloc[0][:300])



Text cleaning completed!

Original review:
I am one of Jehovah's Witnesses and I also work in an acute care medical facility. Over the years I have seen people die from hemolytic reactions to blood transfusions, have attended numerous conferences on blood born pathogens, and have seen several patients become seriously ill from pathogens indu

Cleaned review:
i am one of jehovahs witnesses and i also work in an acute care medical facility over the years i have seen people die from hemolytic reactions to blood transfusions have attended numerous conferences on blood born pathogens and have seen several patients become seriously ill from pathogens induced 


In [18]:

# Build vocabulary using training reviews only
word_counts = Counter()

for review in X_train_clean:
    word_counts.update(review.split())

# Reserve 0 for padding and 1 for unknown words
vocab = {
    "<PAD>": 0,
    "<UNK>": 1
}

# Limit vocabulary to the 20,000 most frequent words
max_vocab_size = 20000

for word, count in word_counts.most_common(max_vocab_size - len(vocab)):
    vocab[word] = len(vocab)

print("Vocabulary created successfully!")
print("Vocabulary size:", len(vocab))

print("\nSample vocabulary entries:")
print(list(vocab.items())[:20])



Vocabulary created successfully!
Vocabulary size: 20000

Sample vocabulary entries:
[('<PAD>', 0), ('<UNK>', 1), ('the', 2), ('and', 3), ('a', 4), ('of', 5), ('to', 6), ('is', 7), ('in', 8), ('it', 9), ('i', 10), ('this', 11), ('that', 12), ('was', 13), ('as', 14), ('for', 15), ('with', 16), ('movie', 17), ('but', 18), ('film', 19)]


In [19]:

# Maximum number of tokens per review
MAX_LEN = 200


def tokenize_review(review, vocab, max_len=MAX_LEN):
    # Convert words into token IDs
    tokens = [
        vocab.get(word, vocab["<UNK>"])
        for word in review.split()
    ]

    # Truncate reviews that are too long
    tokens = tokens[:max_len]

    # Pad shorter reviews with zeros
    tokens += [vocab["<PAD>"]] * (max_len - len(tokens))

    return tokens


# Convert training and testing reviews
X_train_seq = [
    tokenize_review(review, vocab)
    for review in X_train_clean
]

X_test_seq = [
    tokenize_review(review, vocab)
    for review in X_test_clean
]


print("Tokenization and padding completed!")

print("\nMaximum sequence length:", MAX_LEN)
print("Training sequences:", len(X_train_seq))
print("Testing sequences:", len(X_test_seq))

print("\nFirst 20 token IDs:")
print(X_train_seq[0][:20])

print("\nLength of first training sequence:", len(X_train_seq[0]))
print("Length of first testing sequence:", len(X_test_seq[0]))



Tokenization and padding completed!

Maximum sequence length: 200
Training sequences: 39665
Testing sequences: 9917

First 20 token IDs:
[10, 229, 28, 5, 1, 4905, 3, 10, 81, 161, 8, 33, 1, 446, 2721, 7710, 129, 2, 151, 10]

Length of first training sequence: 200
Length of first testing sequence: 200


In [21]:

# Convert sequences and labels into PyTorch tensors
X_train_tensor = torch.tensor(X_train_seq, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_seq, dtype=torch.long)

y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.to_numpy(), dtype=torch.float32)


# Create a custom Dataset
class SentimentDataset(Dataset):
    def __init__(self, reviews, labels):
        self.reviews = reviews
        self.labels = labels

    def __len__(self):
        return len(self.reviews)

    def __getitem__(self, index):
        return self.reviews[index], self.labels[index]


# Create training and testing datasets
train_dataset = SentimentDataset(X_train_tensor, y_train_tensor)
test_dataset = SentimentDataset(X_test_tensor, y_test_tensor)


# Create DataLoaders
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


print("PyTorch datasets and DataLoaders created successfully!")
print("\nTraining samples:", len(train_dataset))
print("Testing samples:", len(test_dataset))
print("Batch size:", BATCH_SIZE)
print("Training batches:", len(train_loader))
print("Testing batches:", len(test_loader))


# Check one batch
sample_reviews, sample_labels = next(iter(train_loader))

print("\nSample batch review shape:", sample_reviews.shape)
print("Sample batch label shape:", sample_labels.shape)
print("First 10 labels in the batch:", sample_labels[:10].tolist())



PyTorch datasets and DataLoaders created successfully!

Training samples: 39665
Testing samples: 9917
Batch size: 64
Training batches: 620
Testing batches: 155

Sample batch review shape: torch.Size([64, 200])
Sample batch label shape: torch.Size([64])
First 10 labels in the batch: [1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0]


In [22]:

# Define the LSTM sentiment classification model
class LSTMSentimentClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(LSTMSentimentClassifier, self).__init__()

        # Convert word IDs into embeddings
        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        # Learn patterns in the sequence of words
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        # Classify the review using the final hidden state
        self.fc = nn.Linear(hidden_dim, 1)

        # Convert output into a probability
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        embedded = self.embedding(x)

        _, (hidden, _) = self.lstm(embedded)

        hidden = hidden[-1]

        output = self.fc(hidden)

        probability = self.sigmoid(output)

        return probability.squeeze(1)


# Define model parameters
VOCAB_SIZE = len(vocab)
EMBEDDING_DIM = 100
HIDDEN_DIM = 128

# Create the model
model = LSTMSentimentClassifier(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM
)

print("LSTM model created successfully!")
print("\nModel architecture:")
print(model)

print("\nVocabulary size:", VOCAB_SIZE)
print("Embedding dimension:", EMBEDDING_DIM)
print("Hidden dimension:", HIDDEN_DIM)



LSTM model created successfully!

Model architecture:
LSTMSentimentClassifier(
  (embedding): Embedding(20000, 100, padding_idx=0)
  (lstm): LSTM(100, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

Vocabulary size: 20000
Embedding dimension: 100
Hidden dimension: 128


In [23]:

# Define the loss function
criterion = nn.BCELoss()

# Define the optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

# Set the training device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move the model to the selected device
model = model.to(device)

print("Loss function and optimizer configured successfully!")
print("\nLoss function:", criterion)
print("Optimizer:", optimizer.__class__.__name__)
print("Learning rate:", optimizer.param_groups[0]["lr"])
print("Training device:", device)



Loss function and optimizer configured successfully!

Loss function: BCELoss()
Optimizer: Adam
Learning rate: 0.001
Training device: cpu


In [ ]:

# Set the number of training epochs
EPOCHS = 3

# Store training results
train_losses = []

print("Starting LSTM model training...\n")

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for reviews, labels in train_loader:
        reviews = reviews.to(device)
        labels = labels.to(device)

        # Clear gradients from the previous batch
        optimizer.zero_grad()

        # Generate predictions
        predictions = model(reviews)

        # Calculate the loss
        loss = criterion(predictions, labels)

        # Calculate gradients
        loss.backward()

        # Update model parameters
        optimizer.step()

        total_loss += loss.item()

    # Calculate average loss for the epoch
    average_loss = total_loss / len(train_loader)
    train_losses.append(average_loss)

    print(
        f"Epoch [{epoch + 1}/{EPOCHS}], "
        f"Training Loss: {average_loss:.4f}"
    )

print("\nTraining completed!")



Starting LSTM model training...

Epoch [1/3], Training Loss: 0.6935
